# Fine-Tuning

Take a pre-trained imaging model and fine-tune it for our specific location guessing task.

# XAI

Take our final tuned model and apply XAI:
- Genreates a heatmap for any given input (image) which has a different meaning depending on the method/option used below
- Option 1: Saliency Map
  - Computes gradients of output with respect to every input (pixel)
  - Higher heatmap value means places where small changes in input pixels have a large impact on output prediction
- Option 2: Grad-CAM (**let's try this one first**)
  - Gradient computed for the final layers of CNN
  - Resulting heatmap is the regions most relevant to a particular output
- Option 3: LIME
  - Can be used if we aren't using a CNN, it doesn't depend on model type
  - Works by changing individual regions of an input image and seeing how the output is affected, giving more importance to regions that change it more.
- Option 4: LRP
  - Shows which individual neurons contriute most to the final output
  - More for low level diagnostics, less ideal for showing stuff to the end user, but maybe we could use it for tuning or just out of curiosity?

Necessary steps before doing XAI:
- Fine tune a pretrained model for our task

Steps to do XAI:
- Try to implement couple of the above methods, starting with Grad-CAM, maybe saliency map second depending on results
- Pass images in and see what the outputs are like
- If we want, we can further tune the model based on what we're seeing, especially on images it gets wrong (like if it thinks an image of London is New York, we could see which features it's identifying and getting wrong (e.g. road markings), and try to have more training images with those feautres, even doing this manually for a handful of images for a certain feature as proof of concept)

Comment on the above:
- Grad-CAM didn't work so we tried another method
- Attention Rollout is used instead, more tailored for transformer models
- Insted of calculating gradients (to find out which small changes in input pixels contribute to the largest changes in output) it measures attention (can be a drawback if the model isn't only attention based)
- It doesn't work by seeing how much a pixel change would alter the output. It instead looks at how much each input (token, which is a block of pixels) contributes to the final decision (path-based, not a derivative)
- The output represents attention scores

In [22]:
!pip install transformers torch pillow captum matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


In [23]:
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch.nn.functional as F

In [ ]:
# ==== CONFIG ====
model_name = "google/vit-base-patch16-224"
image_path = "puppy_image.jpg"

# ==== LOAD MODEL AND PROCESSOR ====
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)
model.eval()

# ==== CAPTURE ATTENTION WEIGHTS ====
attn_weights = []

def hook_attention(module, input, output):
    attn_weights.append(output[1].detach())  # attention weights

for block in model.vit.encoder.layer:
    block.attention.attention.register_forward_hook(hook_attention)


# ==== LOAD & PREPROCESS IMAGE ====
image = Image.open(image_path).convert("RGB")
inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    _ = model(**inputs, output_attentions=True)

# ==== PROCESS ATTENTION ROLLOUT ====
# Stack attentions: shape [layers, batch, heads, tokens, tokens]
attn_stack = torch.stack(attn_weights)  # [L, B, H, T, T]
attn_stack = attn_stack.squeeze(1)      # [L, H, T, T]

# Average heads in each layer → [L, T, T]
attn_avg = attn_stack.mean(dim=1)

# Add residual connection to each attention matrix (identity matrix)
num_tokens = attn_avg.size(-1)
identity = torch.eye(num_tokens)
attn_augmented = attn_avg + identity

# Normalize rows
attn_normalized = attn_augmented / attn_augmented.sum(dim=-1, keepdim=True)

# Attention rollout: multiply attention matrices from last to first
rollout = attn_normalized[0]
for i in range(1, attn_normalized.size(0)):
    rollout = torch.matmul(attn_normalized[i], rollout)

# Get attention map for class token to patch tokens
class_attention = rollout[0, 1:]  # remove CLS token, keep [num_patches]

# ==== CONVERT TO HEATMAP ====
# ViT uses 14x14 patches for 224x224 input
grid_size = int(np.sqrt(class_attention.shape[0]))
attn_map = class_attention.reshape(grid_size, grid_size).numpy()
attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

# Upsample to image size
attn_map_tensor = torch.tensor(attn_map).unsqueeze(0).unsqueeze(0)  # [1, 1, 14, 14]
upsampled = F.interpolate(attn_map_tensor, size=(224, 224), mode="bilinear", align_corners=False)
upsampled = upsampled.squeeze().numpy()

# ==== VISUALIZE ====
# Prepare original image
image_resized = image.resize((224, 224))
img_np = np.array(image_resized)

# Create colored heatmap
cmap = cm.get_cmap('jet')
heatmap = cmap(upsampled)[:, :, :3]
heatmap = (heatmap * 255).astype(np.uint8)

# Overlay
overlay = (0.6 * img_np + 0.4 * heatmap).astype(np.uint8)

# Show and save
plt.figure(figsize=(6, 6))
plt.imshow(overlay)
plt.axis("off")
plt.title("Attention Rollout (ViT)")
plt.tight_layout()
plt.savefig("attention_rollout_overlay.png")
plt.show()

testing below with some very rough code for CNN xAI